# Segment 5: Code Generation and Explanation

## Claude API for Python Developers

In this segment, we'll cover:
- Generating Code from Prompts
- Explaining Existing Code
- Generating Comments and Documentation
- Best Practices for Code-Related Tasks


## Setup and Imports


In [ ]:
import anthropic
from IPython.display import display, Markdown, Code

# Initialize the client
client = anthropic.Anthropic()

def show_code(code, language="python"):
    """Display code with syntax highlighting."""
    display(Markdown(f"```{language}\n{code}\n```"))

print("Client initialized successfully!")


---
## 1. Generating Code from Prompts

Claude excels at generating code from natural language descriptions. The key is providing clear, specific requirements.


In [ ]:
# Basic code generation
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are an expert Python developer. Write clean, well-documented code.",
    messages=[
        {
            "role": "user",
            "content": "Write a Python function that checks if a string is a valid email address using regex."
        }
    ]
)

print("📝 Generated Code:")
display(Markdown(response.content[0].text))


In [ ]:
# More specific requirements lead to better code
detailed_prompt = """
Write a Python class called `RateLimiter` with the following specifications:

1. Constructor takes `max_requests` (int) and `time_window` (int, seconds)
2. Method `is_allowed()` returns True if request is within rate limit
3. Method `reset()` clears the request history
4. Use a sliding window algorithm
5. Include type hints
6. Add docstrings

Example usage:
```python
limiter = RateLimiter(max_requests=5, time_window=60)
if limiter.is_allowed():
    # Process request
```
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    system="You are an expert Python developer. Write production-quality code.",
    messages=[{"role": "user", "content": detailed_prompt}]
)

print("📝 Generated Rate Limiter:")
display(Markdown(response.content[0].text))


### Generating Code in Multiple Languages


In [ ]:
# Claude can generate code in many languages
languages = ["Python", "JavaScript", "TypeScript", "Rust", "Go"]

task = "Write a function to reverse a string"

print("🌐 Same task in multiple languages:\n")

for lang in languages:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"{task}. Use {lang}. Just the code, no explanation."
        }]
    )
    
    print(f"📌 {lang}:")
    print(response.content[0].text)
    print("\n" + "-"*50 + "\n")


In [ ]:
# Generating SQL queries from natural language
sql_prompt = """
I have a database with the following tables:
- users (id, name, email, created_at, subscription_tier)
- orders (id, user_id, total_amount, status, created_at)
- products (id, name, price, category)
- order_items (order_id, product_id, quantity)

Write SQL queries for:
1. Get top 10 customers by total spending
2. Find products that have never been ordered
3. Calculate monthly revenue for the past year
4. Find users who haven't ordered in the last 30 days
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    system="You are a database expert. Write efficient, readable SQL queries.",
    messages=[{"role": "user", "content": sql_prompt}]
)

print("📊 Generated SQL Queries:")
display(Markdown(response.content[0].text))


---
## 2. Explaining Existing Code

Claude can analyze and explain code at various levels of detail.


In [ ]:
# Complex code to explain
complex_code = '''
def memoize(func):
    cache = {}
    def wrapper(*args, **kwargs):
        key = str(args) + str(sorted(kwargs.items()))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    return wrapper

@memoize
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Explain this Python code in detail. Cover:
1. What each part does
2. How the decorator pattern works
3. Why memoization helps here
4. The time complexity improvement

```python
{complex_code}
```"""
    }]
)

print("📚 Code Explanation:")
display(Markdown(response.content[0].text))


In [ ]:
# Explaining code for different audiences
async_code = '''
async def fetch_all(urls):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_one(session, url) for url in urls]
        return await asyncio.gather(*tasks)

async def fetch_one(session, url):
    async with session.get(url) as response:
        return await response.json()
'''

audiences = [
    ("beginner", "Explain to someone new to programming. Use simple analogies."),
    ("intermediate", "Explain to someone familiar with Python but new to async."),
    ("expert", "Focus on performance implications and best practices.")
]

print("👥 Same code, different audiences:\n")

for level, instruction in audiences:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""{instruction}

```python
{async_code}
```"""
        }]
    )
    
    print(f"📖 {level.upper()} Explanation:")
    print(response.content[0].text[:500] + "..." if len(response.content[0].text) > 500 else response.content[0].text)
    print("\n" + "="*60 + "\n")


In [ ]:
# Code review and improvement suggestions
code_to_review = '''
def get_user_data(user_id):
    connection = mysql.connector.connect(host="localhost", user="root", password="password123", database="mydb")
    cursor = connection.cursor()
    cursor.execute("SELECT * FROM users WHERE id = " + str(user_id))
    result = cursor.fetchone()
    return result
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1200,
    messages=[{
        "role": "user",
        "content": f"""Review this code for:
1. Security vulnerabilities
2. Best practices violations
3. Error handling issues
4. Provide an improved version

```python
{code_to_review}
```"""
    }]
)

print("🔍 Code Review:")
display(Markdown(response.content[0].text))


---
## 3. Generating Comments and Documentation

Claude can add documentation at various levels: inline comments, docstrings, and full documentation.


In [ ]:
# Code that needs documentation
undocumented_code = '''
class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = None
        self._cache = {}
    
    def load(self, filepath):
        with open(filepath, 'r') as f:
            self.data = json.load(f)
        return self
    
    def transform(self, operations):
        if self.data is None:
            raise ValueError("No data loaded")
        
        result = self.data.copy()
        for op in operations:
            op_name = op['type']
            if op_name in self._cache:
                result = self._cache[op_name](result, op.get('params', {}))
            else:
                handler = getattr(self, f'_op_{op_name}', None)
                if handler:
                    self._cache[op_name] = handler
                    result = handler(result, op.get('params', {}))
        return result
    
    def _op_filter(self, data, params):
        field = params['field']
        value = params['value']
        return [d for d in data if d.get(field) == value]
    
    def _op_sort(self, data, params):
        return sorted(data, key=lambda x: x.get(params['field'], ''))
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=2000,
    messages=[{
        "role": "user",
        "content": f"""Add comprehensive documentation to this code:
1. Class-level docstring (purpose, usage example)
2. Method docstrings (Google style with Args, Returns, Raises)
3. Inline comments for complex logic
4. Type hints

```python
{undocumented_code}
```

Return the fully documented code."""
    }]
)

print("📝 Documented Code:")
display(Markdown(response.content[0].text))


In [ ]:
# Generate different documentation styles
simple_function = '''
def calculate_discount(price, discount_percent, is_member=False):
    base_discount = price * (discount_percent / 100)
    if is_member:
        base_discount *= 1.1
    final_price = price - base_discount
    return max(0, final_price)
'''

doc_styles = [
    ("Google Style", "Use Google-style docstrings"),
    ("NumPy Style", "Use NumPy-style docstrings"),
    ("Sphinx Style", "Use Sphinx/reStructuredText style docstrings")
]

print("📚 Different Documentation Styles:\n")

for style_name, instruction in doc_styles:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""{instruction} for this function:

```python
{simple_function}
```

Just show the function with the docstring, no explanation."""
        }]
    )
    
    print(f"📌 {style_name}:")
    print(response.content[0].text)
    print("\n" + "="*60 + "\n")


In [ ]:
# Generate README documentation
project_code = '''
# main.py
from dataclasses import dataclass
import requests

@dataclass
class WeatherData:
    temperature: float
    humidity: int
    description: str

class WeatherClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.weather.com/v1"
    
    def get_current(self, city: str) -> WeatherData:
        response = requests.get(f"{self.base_url}/current", params={"city": city, "key": self.api_key})
        data = response.json()
        return WeatherData(data["temp"], data["humidity"], data["desc"])
    
    def get_forecast(self, city: str, days: int = 5) -> list[WeatherData]:
        response = requests.get(f"{self.base_url}/forecast", params={"city": city, "days": days, "key": self.api_key})
        return [WeatherData(d["temp"], d["humidity"], d["desc"]) for d in response.json()["days"]]
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Generate a professional README.md for this Python library:

```python
{project_code}
```

Include:
1. Project title and description
2. Installation instructions
3. Quick start example
4. API reference
5. Contributing section
6. License placeholder"""
    }]
)

print("📄 Generated README:")
display(Markdown(response.content[0].text))


---
## 4. Advanced Code Tasks

### Converting Between Languages


In [ ]:
# Convert Python to JavaScript
python_code = '''
class ShoppingCart:
    def __init__(self):
        self.items = []
    
    def add_item(self, name, price, quantity=1):
        self.items.append({
            "name": name,
            "price": price,
            "quantity": quantity
        })
    
    def get_total(self):
        return sum(item["price"] * item["quantity"] for item in self.items)
    
    def apply_discount(self, percent):
        discount = self.get_total() * (percent / 100)
        return self.get_total() - discount
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Convert this Python class to modern JavaScript (ES6+):

```python
{python_code}
```

Use classes, arrow functions, and modern JS best practices."""
    }]
)

print("🔄 Python → JavaScript Conversion:")
display(Markdown(response.content[0].text))


### Generating Unit Tests


In [ ]:
# Generate tests for a function
function_to_test = '''
def validate_password(password: str) -> dict:
    """
    Validate a password against security rules.
    Returns a dict with 'valid' (bool) and 'errors' (list).
    """
    errors = []
    
    if len(password) < 8:
        errors.append("Password must be at least 8 characters")
    
    if not any(c.isupper() for c in password):
        errors.append("Password must contain at least one uppercase letter")
    
    if not any(c.islower() for c in password):
        errors.append("Password must contain at least one lowercase letter")
    
    if not any(c.isdigit() for c in password):
        errors.append("Password must contain at least one number")
    
    if not any(c in "!@#$%^&*" for c in password):
        errors.append("Password must contain at least one special character (!@#$%^&*)")
    
    return {"valid": len(errors) == 0, "errors": errors}
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Generate comprehensive pytest unit tests for this function:

```python
{function_to_test}
```

Include:
1. Test valid passwords
2. Test each error condition
3. Edge cases
4. Parametrized tests where appropriate"""
    }]
)

print("🧪 Generated Unit Tests:")
display(Markdown(response.content[0].text))


### Refactoring Code


In [ ]:
# Refactor messy code
messy_code = '''
def process(d):
    r = []
    for i in d:
        if i['type'] == 'A':
            x = i['value'] * 2
            if x > 100:
                x = 100
            r.append({'id': i['id'], 'result': x, 'category': 'doubled'})
        elif i['type'] == 'B':
            x = i['value'] + 10
            if x > 100:
                x = 100
            r.append({'id': i['id'], 'result': x, 'category': 'incremented'})
        elif i['type'] == 'C':
            x = i['value'] / 2
            if x < 0:
                x = 0
            r.append({'id': i['id'], 'result': x, 'category': 'halved'})
    return r
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[{
        "role": "user",
        "content": f"""Refactor this code to be cleaner and more maintainable:

```python
{messy_code}
```

Apply:
1. Meaningful variable names
2. Extract helper functions
3. Use type hints
4. Add docstrings
5. Apply SOLID principles where appropriate
6. Use modern Python features (e.g., dataclasses, match statements)"""
    }]
)

print("✨ Refactored Code:")
display(Markdown(response.content[0].text))


### Debugging Assistance


In [ ]:
# Debug code with an error
buggy_code = '''
def find_duplicates(items):
    """Find duplicate items in a list."""
    seen = {}
    duplicates = []
    
    for item in items:
        if item in seen:
            duplicates.append(item)
        seen[item] = True
    
    return duplicates

# Expected: [2, 3]
# Actual: [2, 2, 3]
result = find_duplicates([1, 2, 2, 2, 3, 3, 4])
'''

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": f"""Debug this code. The expected output is [2, 3] but it returns [2, 2, 3]:

```python
{buggy_code}
```

1. Identify the bug
2. Explain why it happens
3. Provide the fixed code"""
    }]
)

print("🐛 Debug Analysis:")
display(Markdown(response.content[0].text))


---
## Best Practices for Code Generation

### Tips for Better Results:

1. **Be Specific**: Include requirements, constraints, and expected behavior
2. **Provide Context**: Share relevant code, schemas, or examples
3. **Request Explanations**: Ask Claude to explain its choices
4. **Iterate**: Use follow-up prompts to refine the code
5. **Verify**: Always test generated code before using it

### Effective Prompt Patterns:

```python
# Pattern 1: Requirements first
"Create a function that:
- Takes X as input
- Returns Y
- Handles edge cases A, B, C
- Uses library Z"

# Pattern 2: Example-driven
"Write code that transforms this input:
[example input]
Into this output:
[example output]"

# Pattern 3: Style specification
"Using [language/framework], following [style guide], 
implement [feature] with [constraints]"
```


---
## Summary

In this segment, we covered:

1. **Generating Code**: From natural language descriptions to working code
2. **Explaining Code**: At different levels for different audiences
3. **Documentation**: Docstrings, comments, and README generation
4. **Advanced Tasks**: Language conversion, testing, refactoring, debugging

### Key Takeaways:
- Specific prompts yield better code
- Include requirements, constraints, and examples
- Use system prompts to set coding standards
- Always review and test generated code
- Claude can adapt to different styles and audiences

### Best Practices:
- Provide context (existing code, schemas, requirements)
- Request explanations to understand the code
- Use iterative refinement for complex tasks
- Specify language version and style guides
- Include edge cases in your prompts

---

## Course Wrap-Up

Congratulations! You've completed the Claude API for Python Developers course.

### What We Covered:

1. **Segment 1**: Generative AI fundamentals and API basics
2. **Segment 2**: Claude models, prompting, and summarization
3. **Segment 3**: Embeddings with Voyage AI for semantic search
4. **Segment 4**: Tools for structured outputs and agents
5. **Segment 5**: Code generation and explanation

### Next Steps:
- Build a project using what you've learned
- Explore the Anthropic documentation for advanced features
- Join the Anthropic community for updates and support

Happy coding with Claude! 🚀
